### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [3]:
from langchain_groq import ChatGroq

model = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

In [4]:
response=model.invoke("Why do parrots talk?")
print(response.content)

**Short answer:**  
Parrots “talk” because they are *vocal learners*—they have the brain circuitry and the physical organ (the syrinx) that let them imitate sounds. In the wild they use this ability to communicate with their flock, to signal danger, to attract mates, and to bond socially. In captivity, the same skill is redirected toward humans, who provide food, attention, and a social partner, so parrots learn to mimic our words and sounds.

---

## 1.  The biology that makes parrots talk

| Feature | What it does | Why it matters |
|---------|--------------|----------------|
| **Syrinx** | The bird’s vocal organ, located at the base of the trachea. | Parrots can shape the syrinx in many ways, producing a wide range of sounds. |
| **Vocal‑learning brain nuclei** | Specialized nuclei (e.g., the robust nucleus of the arcopallium, HVC, and the lateral magnocellular nucleus of the nidopallium) that connect the auditory cortex to the syrinx. | These nuclei allow parrots to *learn* sounds 

In [6]:
from langchain.tools import tool
@tool
def get_weather(location:str)->str:
    """Get the current weather in a given location."""
    return f"The current weather in {location} is sunny with a high of 75 degrees."
model_with_tools=model.bind_tools([get_weather])


In [13]:
response=model_with_tools.invoke("What is the weather in New York?")
print(response)

for tool_calls in response.tool_calls:
    print(f"Tool Name: {tool_calls['name']}")
    print(f"Tool Args: {tool_calls['args']}")


content='' additional_kwargs={'reasoning_content': 'We need to call get_weather function.', 'tool_calls': [{'id': 'fc_b295a3eb-e8f1-431e-8aaf-0d3179f5a368', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 130, 'total_tokens': 163, 'completion_time': 0.034628381, 'completion_tokens_details': {'reasoning_tokens': 9}, 'prompt_time': 0.01750459, 'prompt_tokens_details': None, 'queue_time': 0.278679862, 'total_time': 0.052132971}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_ef00694abe', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a060db-e7d7-70b3-b992-e2ea20b172c8-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'New York'}, 'id': 'fc_b295a3eb-e8f1-431e-8aaf-0d3179f5a368', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 130, 'output_tok

### Tool execution loop

In [14]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

The current weather in Boston is sunny with a high of 75 °F.


In [15]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call get_weather function.', 'tool_calls': [{'id': 'fc_629ff9f8-c613-4026-8feb-a3e6c5a81724', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 128, 'total_tokens': 160, 'completion_time': 0.049700647, 'completion_tokens_details': {'reasoning_tokens': 9}, 'prompt_time': 0.007592188, 'prompt_tokens_details': None, 'queue_time': 0.285427377, 'total_time': 0.057292835}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_4f7e7dc26e', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a060dc-c090-7391-9994-e280f8e70426-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_629ff9f8-c613-4026-8feb-a3e6c5a81724', 'type': 'tool_cal